# <h1 style="text-align: center;">1.DATA CLEANING</h1>


## 1. Cleaning responsivenes table

## Reading csv file
We start by loading the raw responsiveness data into a DataFrame so we can inspect and clean it.

In [1]:
import pandas as pd

df_responsivenes = pd.read_csv("responsivenes.csv")
df_responsivenes.info()


<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   inpatient_number  2008 non-null   int64
 1   eye_opening       2008 non-null   int64
 2   verbal_response   2008 non-null   int64
 3   movement          2008 non-null   int64
 4   consciousness     2008 non-null   str  
 5   gcs               2008 non-null   int64
dtypes: int64(5), str(1)
memory usage: 104.4 KB


## Renaming columns
Raw column name inpatient_number isn't clear or consistent with how we want to refer to patients throughout this notebook. We rename it to patient_id so the column is easier to understand and so it matches the naming used across our other tables. The remaining columns (eye_opening, verbal_response, movement, consciousness, gcs) already have clear, self-explanatory names, so no further renaming is needed.

In [2]:
# Mapping old column name to new column name
df_responsivenes = df_responsivenes.rename(columns={
    'inpatient_number': 'patient_id'
})

In [3]:
df_responsivenes

,patient_id,eye_opening,verbal_response,movement,consciousness,gcs
0,857781,4,5,6,Clear,15
1,743087,4,5,6,Clear,15
2,866418,4,5,6,Clear,15
3,775928,4,5,6,Clear,15
4,810128,4,5,6,Clear,15
...,...,...,...,...,...,...
2003,740689,4,5,6,Clear,15
2004,734280,4,5,6,Clear,15
2005,781004,4,5,6,Clear,15
2006,744870,1,1,1,Nonresponsive,3


## Checking and Standardizing column values
Text columns often contain inconsistent formatting — extra spaces, mismatched capitalization, or missing entries — that can make the same real-world value look like different categories to pandas. `consciousness` is the only text column here, so we check it for stray whitespace. Note that we deliberately do **not** apply `.str.title()` the way we did for the demography table for example: `consciousness` values like `ResponsiveToPain` are <i>compound CamelCase labels, and title-casing would incorrectly lowercase everything after the first letter (turning it into `Responsivetopain`). Stripping whitespace is enough to standardize this column safely.</i>

In [4]:
# Clean text directly on the object series
df_responsivenes['consciousness'] = df_responsivenes['consciousness'].str.strip()

# Check unique values
print(df_responsivenes['consciousness'].unique())

# Expected ideal output: ['Clear', 'ResponsiveToPain', 'ResponsiveToSound', 'Nonresponsive']
# There are no NaN, extra whitespace, or inconsistent casing, and so we do note need to clean it further.

<ArrowStringArray>
['Clear', 'ResponsiveToPain', 'ResponsiveToSound', 'Nonresponsive']
Length: 4, dtype: str


## Check for data types
We check how pandas has inferred each column's type, then <i>convert `consciousness` to the `category` dtype, which is more memory-efficient</i> and semantically correct for a column with a fixed set of repeating values. `eye_opening`, `verbal_response`, `movement`, and `gcs` <i>are already whole-number scores with no missing values, so `int64` is the correct dtype for them as-is.</i>

In [5]:
df_responsivenes.info()

<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   patient_id       2008 non-null   int64
 1   eye_opening      2008 non-null   int64
 2   verbal_response  2008 non-null   int64
 3   movement         2008 non-null   int64
 4   consciousness    2008 non-null   str  
 5   gcs              2008 non-null   int64
dtypes: int64(5), str(1)
memory usage: 104.7 KB


In [6]:
df_responsivenes['consciousness'] = df_responsivenes['consciousness'].astype('category')

In [7]:
df_responsivenes.info()

<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   patient_id       2008 non-null   int64   
 1   eye_opening      2008 non-null   int64   
 2   verbal_response  2008 non-null   int64   
 3   movement         2008 non-null   int64   
 4   consciousness    2008 non-null   category
 5   gcs              2008 non-null   int64   
dtypes: category(1), int64(5)
memory usage: 80.6 KB


## Check for duplicate records
Duplicate rows would double-count a patient's consciousness assessment and bias any statistics or models built from this table. We check for exact duplicate rows before proceeding.<i> No duplicate rows were found as seen in the output below.</i>

In [8]:
# Count total exact duplicate rows
duplicate_count = df_responsivenes.duplicated().sum()
print("Total fully duplicate rows:", duplicate_count)

Total fully duplicate rows: 0


## Checking for Outliers
The Glasgow Coma Scale (GCS) components each have a fixed, clinically-defined valid range, so — just like the demography table — we check each column against standard clinical reference ranges rather than a statistical method like IQR. We also run a formula consistency check: `gcs` is defined as `eye_opening + verbal_response + movement`, so a recorded `gcs` that doesn't match the sum of its own components would signal a data entry error, even if each individual value looks "in range" on its own.
<i>As seen in the output below there were no outliers or formula mismatches found and so no further cleaning action in this aspect is required.</i>

In [9]:
import pandas as pd

id_col = 'patient_id'

# Standard clinical reference ranges for each GCS component
CLINICAL_RANGES = {
    'eye_opening':      {'min': 1, 'max': 4,  'unit': 'points'},
    'verbal_response':  {'min': 1, 'max': 5,  'unit': 'points'},
    'movement':         {'min': 1, 'max': 6,  'unit': 'points'},
    'gcs':              {'min': 3, 'max': 15, 'unit': 'points'}
}

range_outlier_records = []
formula_mismatch_records = []

# 1. Range Check (eye_opening, verbal_response, movement, gcs)
for col, bounds in CLINICAL_RANGES.items():
    if col in df_responsivenes.columns:
        # Flag values strictly outside plausible clinical boundaries
        mask = (df_responsivenes[col] < bounds['min']) | (df_responsivenes[col] > bounds['max'])

        for idx in df_responsivenes[mask].index:
            range_outlier_records.append({
                'patient_id': df_responsivenes.loc[idx, id_col],
                'variable': col,
                'outlier_value': df_responsivenes.loc[idx, col],
                'issue': f"Outside clinical range ({bounds['min']}–{bounds['max']} {bounds['unit']})"
            })

# 2. Formula Consistency Check (Calculated GCS vs Recorded GCS)
if all(col in df_responsivenes.columns for col in ['eye_opening', 'verbal_response', 'movement', 'gcs']):
    valid_rows = df_responsivenes.dropna(subset=['eye_opening', 'verbal_response', 'movement', 'gcs']).copy()
    calc_gcs = valid_rows['eye_opening'] + valid_rows['verbal_response'] + valid_rows['movement']

    # Flag any discrepancy at all (GCS components must sum exactly)
    mismatch_mask = valid_rows['gcs'] != calc_gcs

    for idx in valid_rows[mismatch_mask].index:
        formula_mismatch_records.append({
            'patient_id': valid_rows.loc[idx, id_col],
            'variable': 'gcs_mismatch',
            'outlier_value': valid_rows.loc[idx, 'gcs'],
            'issue': f"Recorded GCS differs from eye_opening+verbal_response+movement sum ({calc_gcs.loc[idx]})"
        })

# 3. Output Outlier Summary -- range check and formula check reported separately
range_outliers_df = pd.DataFrame(range_outlier_records)
formula_mismatches_df = pd.DataFrame(formula_mismatch_records)

print("=== RANGE CHECK ===")
if not range_outliers_df.empty:
    print(f"DETECTED {len(range_outliers_df)} VALUES OUTSIDE CLINICAL RANGE")
    print(range_outliers_df.to_string(index=False))
else:
    print("No values found outside the clinical reference ranges.")

print("\n=== FORMULA CONSISTENCY CHECK ===")
if not formula_mismatches_df.empty:
    print(f"DETECTED {len(formula_mismatches_df)} GCS FORMULA MISMATCHES")
    print(formula_mismatches_df.to_string(index=False))
else:
    print("No formula mismatches found: gcs equals eye_opening + verbal_response + movement for every row.")

=== RANGE CHECK ===
No values found outside the clinical reference ranges.

=== FORMULA CONSISTENCY CHECK ===
No formula mismatches found: gcs equals eye_opening + verbal_response + movement for every row.


## Checking for Missing values
We check every column for missing values to understand exactly where, if anywhere, data is absent.<i> As indicated by the output, there are no missing values found in any column.</i>

In [10]:
# Total missing values per column
missing_counts = df_responsivenes.isna().sum()
print(missing_counts)

patient_id         0
eye_opening        0
verbal_response    0
movement           0
consciousness      0
gcs                0
dtype: int64


In [11]:
#import os

# Create an output folder for cleaned files (keeps them separate from raw data)
#output_folder = "cleaned"
#os.makedirs(output_folder, exist_ok=True)

# Save the cleaned dataframe
#df_responsivenes.to_csv(f"{output_folder}/responsivenes_cleaned.csv", index=False)

#print("Cleaned file saved to:", output_folder)